In [1]:
do = False # @param{type:"boolean"}
if do:
    %pip install torchinfo -qq
    %pip install -U git+https://github.com/qubvel/segmentation_models.pytorch -qq
    %pip install starfile -qq
    %pip install https://github.com/soft-matter/trackpy/archive/master.zip -qq

In [2]:
if do:
    !git clone https://github.com/phonchi/CryoParticleSegment.git

In [3]:
import sys
import os

# Adjust the path relative to your current working directory
module_path = os.path.abspath('CryoParticleSegment/Modeling')

# Add to sys.path if it's not already included
if module_path not in sys.path:
    sys.path.append(module_path)

In [4]:
if do:
    #%git clone https://github.com/netw0rkf10w/CRF.git
    %cd CryoParticleSegment/Modeling/CRF_main
    !python setup.py clean --all
    !rm -rf build/
    !python setup.py build_ext --inplace --force
    !python setup.py install

In [5]:
if do:
    %pip install pycuda==2024.1
    %pip install "numpy<2.0"
    %pip install mrcfile -qq

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
# @title  { display-mode: "form" }

IMAGE_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/dataset/10017/processed_micrographs_np_split" # @param {type:"string"}
LABEL_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output/dataset/10017/micrographs_ground_np" # @param {type:"string"}
RESULT_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/results_100_CDF/10017/unet_eb5_dice_CRF" # @param {type:"string"}

import os
if "content" in IMAGE_DIR.split("/")[:3] or "content" in LABEL_DIR.split("/")[:3]:
  try:
    from google.colab import drive
    drive.mount('/content/drive')
    !rm -r /content/sample_data
    if not os.path.exists("/content/image_dir"):
        if "content" in LABEL_DIR.split("/")[:3]:
            !cp -r {LABEL_DIR} /content/label_dir
            LABEL_DIR = "/content/label_dir"
  except:
    pass

%cd /content/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content


In [8]:
# @title  { display-mode: "form" }
# @markdown Useful packages.

import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau, OneCycleLR
import gc
from tqdm import tqdm

In [9]:
# @title  { display-mode: "form" }
# @markdown User-defined packages.

from dataset import MicrographDataset, MicrographDatasetEvery
from dataset import reconstruct_patched
from model import create_model
from trainer import CryoEMEvaluator
from trainer import CryoEMTrainerWithScheduler, tqdm_plugin_for_Trainer

In [10]:
def simple_micrograph_preprocessing(micrograph):
    micrograph_copy = micrograph.copy()
    micrograph_copy = (micrograph_copy-micrograph.mean()+2.5*micrograph.std())/5/micrograph.std()
    micrograph_copy[micrograph_copy<0]=0
    micrograph_copy[micrograph_copy>1]=1
    return micrograph_copy

def normalize(im):
    max_mrc=np.max(im)
    min_mrc=np.min(im)
    img_original=(255*((im-min_mrc)/(max_mrc-min_mrc))).astype(np.uint8)
    return(img_original)

def preprocess_and_crop(micrograph, crop_size=3840):
    processed_micrograph = simple_micrograph_preprocessing(micrograph)
    if crop_size:
        mic_width, mic_height = processed_micrograph.shape[1], processed_micrograph.shape[0]
        start_x, start_y = (mic_width - crop_size) // 2, (mic_height - crop_size) // 2
        end_x, end_y = start_x + crop_size, start_y + crop_size
        return processed_micrograph[start_y:end_y, start_x:end_x]
    else:
        return processed_micrograph

In [11]:
# @markdown Parameters.

NUM_CLASSES = 2
EPOCHS = 50
BATCH = 8
CROP_SIZE = (512, 512)
LR = 1e-3

RLR_PATIENCE = 3
ES_PATIENCE = 15
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [12]:
# @markdown Set seed.

random_state = 42
torch.manual_seed(random_state)
torch.cuda.manual_seed_all(random_state)

In [13]:
# @title  { display-mode: "form" }

architecture = "Unet++" # @param {type:"string"}
encoder = "timm-efficientnet-b5" # @param {type:"string"}
pretrained = True # @param {type:"boolean"}
solver = "fw" # @param {type:"string"}
use_unary_only = True # @param {type:"boolean"}


In [14]:
import segmentation_models_pytorch as smp

if pretrained:
  weights = "imagenet"
else:
  weights = None

if architecture == "Unet++":
    backbone = smp.UnetPlusPlus(
        encoder_name=encoder,        # choose encoder, densenet201, resnet50, e.g. mobilenet_v2 or efficientnet-b5
        encoder_weights=weights,     # use `imagenet` or `advprop` for pre-trained weights for encoder initialization
        in_channels=1,                  # model input channels (1 for gray-scale images, 3 for RGB, etc.)
        classes=2,                      # model output channels (number of classes in your dataset)
    )

elif architecture == "Deeplab":
    backbone = smp.DeepLabV3(
        encoder_name=encoder,        # choose encoder, densenet201, resnet50, e.g. mobilenet_v2 or efficientnet-b5
        encoder_weights=weights,     # use `imagenet` pre-trained weights for encoder initialization
        in_channels=1,                  # model input channels (1 for gray-scale images, 3 for RGB, etc.)
        classes=2,                      # model output channels (number of classes in your dataset)
    )
else:
    print("Architecture not supported")
    raise NotImplementedError

model = create_model(backbone, addout=True) #crf_args

config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/122M [00:00<?, ?B/s]

In [15]:
import CRF
import torch.nn as nn
from model import setup_crf, create_fwcrf_model

# Example usage
solver = 'fw'  # Assuming the solver type is defined

crf = setup_crf(solver, NUM_CLASSES)
model_post = create_fwcrf_model(model.backbone, crf, use_unary_only=True)

CRF solver: fw
x0_weight: 0.0
FrankWolfeParams: 
	 scheme:	 fixed 
	 stepsize:	 1.0 (for the 'fixed' scheme) 
	 regularizer:	 l2
	 lambda_:	 1.0
	 lambda_learnable:	 False
	 x0_weight:	 0.5
	 x0_weight_learnable:	 False
Non-trainable lambda for Frank-Wolfe: 1.0
Non-trainable x0_weight for Frank-Wolfe: 0.5
Potts: remove random weights.
Add 1.0 to spatial_weight diagonal
Add 1.0 to bilateral_weight diagonal
Add -1.0 to compatibility diagonal


In [16]:
from torch.utils.data import ConcatDataset
import torch.nn.functional as F

def run_and_plot_limited_inference(
    image_dir: str,
    label_dir: str,
    result_dir: str,
    model_instance: torch.nn.Module,
    device: torch.device,
    crop_size: tuple,
    max_micrographs: int = 10,
    mini_batch_size: int = 18
):
    """
    Loads data, runs model inference on a limited number of micrographs,
    and plots the resulting mask overlay.
    """

    # --- 1. Setup Data Loaders ---
    print("Setting up datasets...")
    train_dir = os.path.join(image_dir, 'train')
    val_dir = os.path.join(image_dir, 'val')
    test_dir = os.path.join(image_dir, 'test')

    train_filenames = np.loadtxt(f"{image_dir}/train_filenames.txt", dtype=str)
    val_filenames = np.loadtxt(f"{image_dir}/val_filenames.txt", dtype=str)
    test_filenames = np.loadtxt(f"{image_dir}/test_filenames.txt", dtype=str)
    full_filenames = np.concatenate((train_filenames, val_filenames, test_filenames))

    # Assume MicrographDatasetEvery is imported/available
    train_dataset = MicrographDatasetEvery(image_dir=train_dir, label_dir=label_dir, filenames=train_filenames, crop_size=crop_size)
    val_dataset = MicrographDatasetEvery(image_dir=val_dir, label_dir=label_dir, filenames=val_filenames, crop_size=crop_size)
    test_dataset = MicrographDatasetEvery(image_dir=test_dir, label_dir=None, filenames=test_filenames, crop_size=crop_size)
    full_dataset = ConcatDataset([train_dataset, val_dataset, test_dataset])

    # Get the target image shape (full size after reconstruction)
    shape = None
    for _, _, _, i4 in val_dataset:
        shape = i4.shape
        break

    # --- 2. Preprocess and Prepare Background Images (Limited) ---
    print(f"Preprocessing {max_micrographs} background micrographs...")
    n = len(full_dataset)
    processed_micrographs = np.empty((max_micrographs, shape[1], shape[2]), dtype=np.float32)

    for idx, (test_image, _, grid, _) in tqdm(enumerate(full_dataset), total=max_micrographs, desc="Preprocessing images"):
        if idx >= max_micrographs:
            break

        name = full_filenames[idx][:-4]

        for d in ['test', 'train', 'val']:
            micrograph_path = f"{image_dir}/{d}/{name}.npy"
            if os.path.exists(micrograph_path):
                break

        micrograph = np.load(micrograph_path)
        processed_micrograph = preprocess_and_crop(micrograph, crop_size=shape[1])
        processed_micrographs[idx] = processed_micrograph

    # --- 3. Model Setup ---
    print("Loading model and preparing for inference...")
    gc.collect()
    torch.cuda.empty_cache()

    # Load the best model checkpoint
    checkpoint_paths = [path for path in os.listdir(result_dir) if '.pt' in path]
    # Assuming the latest checkpoint is the desired one
    checkpoint_path = checkpoint_paths[-1] if checkpoint_paths else None

    if not checkpoint_path:
        print(f"Error: No checkpoint file found in {result_dir}")
        return

    state_dict_path = f"{result_dir}/{checkpoint_path}"
    state_dict = torch.load(state_dict_path, map_location=device)
    model_instance.load_state_dict(state_dict, strict=False)

    model_instance.to(device)
    model_instance.eval()

    # --- 4. Inference and Plotting Loop (Limited) ---
    print(f"Running inference and plotting {max_micrographs} samples...")

    # Array sized for only MAX_MICROGRAPHS masks
    label_images = np.empty((max_micrographs, shape[1], shape[2]), dtype=np.uint8)

    for idx, (test_image, _, grid, _) in tqdm(enumerate(full_dataset), total=max_micrographs, desc="Processing dataset"):
        if idx >= max_micrographs:
            break

        with torch.no_grad():
            inputs = test_image.to(device)
            num_batches = (inputs.size(0) + mini_batch_size - 1) // mini_batch_size
            patched_outputs = []

            for batch_idx in range(num_batches):
                start_idx = batch_idx * mini_batch_size
                end_idx = min(start_idx + mini_batch_size, inputs.size(0))
                patch_input = inputs[start_idx:end_idx].to(device)
                output = model_instance(patch_input)['out']
                patched_outputs.append(output.cpu())

                del patch_input, output
                torch.cuda.empty_cache()

            outputs = torch.cat(patched_outputs)
            probabilities = F.softmax(outputs, dim=1)
            class1_probabilities = probabilities[:, 1, :, :]

            # Assume 'reconstruct_patched' is available
            pred_image = reconstruct_patched(class1_probabilities.unsqueeze(1), grid).float()
            output_image = normalize(pred_image.squeeze().numpy())

            del patched_outputs, outputs, probabilities, class1_probabilities, pred_image
            torch.cuda.empty_cache()

        label_images[idx] = output_image

        # --- Plotting Code (Plots ALL limited images) ---
        _, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(processed_micrographs[idx], cmap='gray')
        ax.imshow(output_image, cmap='inferno', alpha=0.4)
        ax.set_title(f"Micrograph {idx+1} of {max_micrographs}")
        plt.show()
        # --- End Plotting Code ---

        del output_image
        torch.cuda.empty_cache()

    return label_images

### 1-1

In [17]:
a = run_and_plot_limited_inference(
     image_dir=IMAGE_DIR,
     label_dir=LABEL_DIR,
     result_dir=RESULT_DIR,
     model_instance=model,
     device=device,
     crop_size=CROP_SIZE,
     max_micrographs=10 # Process and plot only the first 10
)

Output hidden; open in https://colab.research.google.com to view.

---

### 1-2

In [18]:
# @title  { display-mode: "form" }

IMAGE_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/dataset/10017/processed_micrographs_np_split" # @param {type:"string"}
RESULT_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/results_100_CDF/10017/unet_eb5_dice" # @param {type:"string"}

%cd /content/

/content


In [19]:
a = run_and_plot_limited_inference(
     image_dir=IMAGE_DIR,
     label_dir=LABEL_DIR,
     result_dir=RESULT_DIR,
     model_instance=model,
     device=device,
     crop_size=CROP_SIZE,
     max_micrographs=10 # Process and plot only the first 10
)

Output hidden; open in https://colab.research.google.com to view.

---

### 2-1

In [20]:
# @title  { display-mode: "form" }

IMAGE_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/time_test_output_tpz/dataset/10017/processed_micrographs_np_split" # @param {type:"string"}
RESULT_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/time_test_output_tpz/results_100_CRF/10017/unet_eb5_dice_CRF" # @param {type:"string"}

%cd /content/

/content


In [21]:
a = run_and_plot_limited_inference(
     image_dir=IMAGE_DIR,
     label_dir=LABEL_DIR,
     result_dir=RESULT_DIR,
     model_instance=model,
     device=device,
     crop_size=CROP_SIZE,
     max_micrographs=10 # Process and plot only the first 10
)

Output hidden; open in https://colab.research.google.com to view.

---
### 2-2

In [22]:
# @title  { display-mode: "form" }

IMAGE_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/time_test_output_tpz/dataset/10017/processed_micrographs_np_split" # @param {type:"string"}
RESULT_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/time_test_output_tpz/results_50_CRF/10017/unet_eb5_dice_CRF" # @param {type:"string"}

%cd /content/

/content


In [23]:
a = run_and_plot_limited_inference(
     image_dir=IMAGE_DIR,
     label_dir=LABEL_DIR,
     result_dir=RESULT_DIR,
     model_instance=model,
     device=device,
     crop_size=CROP_SIZE,
     max_micrographs=10 # Process and plot only the first 10
)

Output hidden; open in https://colab.research.google.com to view.

---
### 3-1

In [24]:
# @title  { display-mode: "form" }

IMAGE_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/C_CSN_D_float_u_output/dataset/10017/processed_micrographs_np_split" # @param {type:"string"}
RESULT_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/C_CSN_D_float_u_output/results_100_CRF/10017/unet_eb5_dice_CRF" # @param {type:"string"}

%cd /content/

/content


In [25]:
a = run_and_plot_limited_inference(
     image_dir=IMAGE_DIR,
     label_dir=LABEL_DIR,
     result_dir=RESULT_DIR,
     model_instance=model,
     device=device,
     crop_size=CROP_SIZE,
     max_micrographs=10 # Process and plot only the first 10
)

Output hidden; open in https://colab.research.google.com to view.

---
### 3-2

In [26]:
# @title  { display-mode: "form" }

IMAGE_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/C_CSN_D_float_u_output/dataset/10017/processed_micrographs_np_split" # @param {type:"string"}
RESULT_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/C_CSN_D_float_u_output/results_50_CRF/10017/unet_eb5_dice_CRF" # @param {type:"string"}

%cd /content/

/content


In [27]:
a = run_and_plot_limited_inference(
     image_dir=IMAGE_DIR,
     label_dir=LABEL_DIR,
     result_dir=RESULT_DIR,
     model_instance=model,
     device=device,
     crop_size=CROP_SIZE,
     max_micrographs=10 # Process and plot only the first 10
)

Output hidden; open in https://colab.research.google.com to view.